# Week 5 Assignment: Building Your First Supervised Machine Learning Model

**Project:** Using Machine Learning to Support Climate Governance and Adaptation Decision-Making  
**Student:** Erick Munala Sifuna

## Objective
This notebook builds and compares two supervised **regression** models using the ND-GAIN Country Index dataset. The target is a country's climate-vulnerability score. Candidate predictors are selected from the available numeric columns while excluding the target and closely related vulnerability-component variables where possible.

The notebook:
1. Loads and inspects the dataset.
2. Identifies the target and input features.
3. Splits observations into training and test sets.
4. Trains Linear Regression and Random Forest Regression.
5. Evaluates predictions using MAE, RMSE, and R².
6. Compares model performance and discusses limitations.

**Important:** The ND-GAIN dataset's column names and downloadable files can change between releases. The loading and column-selection code checks the available columns and prints its choices. Review the selected target and features before interpreting results.

## How to run this notebook
1. Open the notebook in Google Colab or Jupyter Notebook.
2. Run the cells from top to bottom in order.
3. If automatic dataset download fails, upload the relevant CSV/ZIP file as described in the data-loading section.
4. Review the generated metric tables and plots; the final comparison is calculated from the actual test-set results.
5. Save the notebook after running it, then upload the `.ipynb` file to your course-project GitHub repository.


## 1. Import libraries

In [ ]:
import os
import glob
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
RANDOM_STATE = 42

## 2. Load the ND-GAIN dataset
The cell tries to download the ND-GAIN Country Index archive. If that fails, upload the downloaded ZIP or CSV file to the notebook session and run the cell again.

In [ ]:
DATA_URL = "https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip"
ZIP_PATH = "ndgain_countryindex_2026.zip"
EXTRACT_DIR = "ndgain_data"

if not os.path.isdir(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)

if not glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True):
    try:
        if not os.path.exists(ZIP_PATH):
            urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(EXTRACT_DIR)
        print("Downloaded and extracted the ND-GAIN archive.")
    except Exception as exc:
        print("Automatic download did not work:", exc)
        print("Please upload the ND-GAIN ZIP/CSV file and set its path below.")

csv_files = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)
csv_files += [p for p in glob.glob("*.csv") if p not in csv_files]

if not csv_files:
    raise FileNotFoundError(
        "No CSV file was found. Upload the ND-GAIN CSV or ZIP file, "
        "then rerun this section."
    )

tables = {}
for path in csv_files:
    try:
        tables[path] = pd.read_csv(path)
        print(f"{path}: {tables[path].shape[0]} rows x {tables[path].shape[1]} columns")
    except Exception as exc:
        print(f"Could not read {path}: {exc}")

if not tables:
    raise ValueError("CSV files were found, but none could be read.")

# Prefer a table whose filename or columns mention vulnerability/readiness.
def table_score(item):
    path, frame = item
    text = (os.path.basename(path) + " " + " ".join(map(str, frame.columns))).lower()
    return int("vulnerab" in text) * 3 + int("readiness" in text) * 2 + int("country" in text)

selected_path, df = max(tables.items(), key=table_score)
print("\nSelected file:", selected_path)
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

## 3. Identify the target and predictor features
The target is the climate vulnerability score. Column matching is case-insensitive and tolerates common naming variations. Predictor selection avoids country identifiers, year, the target itself, and variables explicitly named as vulnerability components to reduce direct target leakage.

In [ ]:
def normalized_name(name):
    return "".join(ch.lower() for ch in str(name) if ch.isalnum())

column_lookup = {normalized_name(c): c for c in df.columns}

def find_column(candidates):
    # Exact normalized match first
    for candidate in candidates:
        key = normalized_name(candidate)
        if key in column_lookup:
            return column_lookup[key]
    # Then partial match
    for candidate in candidates:
        key = normalized_name(candidate)
        for normalized, original in column_lookup.items():
            if key and key in normalized:
                return original
    return None

target_col = find_column([
    "Vulnerability", "Vulnerability Score", "Vulnerability Index",
    "Climate Vulnerability", "Vulnerability score"
])

if target_col is None:
    possible = [c for c in df.columns if "vulnerab" in str(c).lower()]
    if possible:
        target_col = possible[0]

if target_col is None:
    raise ValueError(
        "Could not identify a vulnerability target column. Inspect df.columns "
        "and set target_col manually to the correct climate-vulnerability score."
    )

# Candidate predictors: readiness is commonly available and relevant to adaptation.
readiness_col = find_column(["Readiness", "Readiness Score", "Readiness Index"])
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# Exclude likely identifiers, time, target, and direct vulnerability components.
excluded_terms = [
    "vulnerab", "exposure", "sensitivity", "adaptivecapacity",
    "countrycode", "iso3", "iso", "year", "rank"
]
feature_cols = []
if readiness_col is not None and readiness_col != target_col and readiness_col in numeric_cols:
    feature_cols.append(readiness_col)

for col in numeric_cols:
    norm = normalized_name(col)
    if col == target_col or col in feature_cols:
        continue
    if any(term in norm for term in excluded_terms):
        continue
    feature_cols.append(col)

# Avoid using a large number of arbitrary columns; prefer readiness plus a small set
# of general numeric indicators. If there are few candidates, use all eligible columns.
feature_cols = feature_cols[:10]

if not feature_cols:
    raise ValueError(
        "No suitable numeric predictors were found. Inspect numeric columns and "
        "manually set feature_cols to relevant indicators."
    )

print("Target/output:", target_col)
print("Predictor/features:", feature_cols)
print("Problem type: regression, because the target is a continuous numerical score.")

# Convert selected fields to numeric and remove rows without a known target.
model_data = df[[target_col] + feature_cols].copy()
for col in model_data.columns:
    model_data[col] = pd.to_numeric(model_data[col], errors="coerce")
model_data = model_data.replace([np.inf, -np.inf], np.nan)
model_data = model_data.dropna(subset=[target_col])

# Remove predictors that contain no usable values.
feature_cols = [c for c in feature_cols if model_data[c].notna().any()]
model_data = model_data[[target_col] + feature_cols]

if len(model_data) < 10:
    raise ValueError("Too few usable rows remain for a reliable train/test split.")

X = model_data[feature_cols]
y = model_data[target_col]

print(f"Usable rows: {len(model_data)}")
display(model_data.head())
display(model_data[[target_col] + feature_cols].describe().T)

## 4. Split into training and test sets
The test set is held back from model fitting so it can provide an estimate of performance on unseen observations.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 5. Train two supervised regression models
Linear Regression provides a simple baseline. Random Forest Regression can model nonlinear patterns and interactions. Missing feature values are imputed using the training data; Linear Regression also standardizes features.

In [ ]:
linear_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2
    ))
])

linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

print("Both models trained successfully.")

## 6. Predict on the test set

In [ ]:
predictions = {
    "Linear Regression": linear_model.predict(X_test),
    "Random Forest": forest_model.predict(X_test)
}

comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Linear Regression": predictions["Linear Regression"],
    "Random Forest": predictions["Random Forest"]
}, index=y_test.index)

display(comparison.head(10))

## 7. Evaluate model performance
Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) measure prediction error; lower values are better. R² measures how much variation in the target is explained by the model; values closer to 1 generally indicate better fit, while a negative test R² means the model performs worse than predicting the test-set mean.

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²": r2_score(y_true, y_pred)
    }

metrics = pd.DataFrame({
    name: evaluate(y_test, pred)
    for name, pred in predictions.items()
}).T

display(metrics.round(4))

## 8. Compare predictions visually

In [ ]:
plt.figure(figsize=(7, 6))
for name, pred in predictions.items():
    plt.scatter(y_test, pred, alpha=0.65, label=name)

low = min(float(y_test.min()), *(float(np.min(p)) for p in predictions.values()))
high = max(float(y_test.max()), *(float(np.max(p)) for p in predictions.values()))
plt.plot([low, high], [low, high], linestyle="--", label="Perfect prediction")
plt.xlabel("Actual climate vulnerability score")
plt.ylabel("Predicted climate vulnerability score")
plt.title("Actual vs predicted climate vulnerability")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Compare models and discuss results

In [ ]:
best_by_rmse = metrics["RMSE"].idxmin()
best_by_mae = metrics["MAE"].idxmin()
best_by_r2 = metrics["R²"].idxmax()

print(f"Lowest RMSE: {best_by_rmse}")
print(f"Lowest MAE: {best_by_mae}")
print(f"Highest test R²: {best_by_r2}")

print("\nInterpretation:")
print("- Lower MAE/RMSE means predictions are closer to the observed scores.")
print("- Higher R² indicates that the model explains more variation in the test data.")
print("- If the models disagree on the best model across metrics, report that trade-off.")
print("- These results show association/predictive performance, not causal effects.")

best_model = metrics["RMSE"].idxmin()
print(f"\nBased on RMSE, {best_model} performed best on this split.")
print("Use the metric table and plot above to support your final discussion.")

## 10. Limitations and responsible interpretation

- ND-GAIN scores are indicators, not complete descriptions of every country's climate risks.
- The result depends on the specific data release, chosen columns, missing-value handling, and random train/test split.
- A random split can place records from the same country in both sets if the dataset contains multiple years per country. For a time-series or country-generalization study, use a time-based or group-based split.
- Some indicators may be conceptually related to the target. Check the printed feature list and exclude any feature that directly contributes to the target score if strict leakage prevention is required.
- Predictive association does not establish causation and should not be used alone to allocate public resources or make policy decisions.

## Conclusion
This notebook completes a supervised learning workflow by defining a numerical target, selecting features, splitting data, training two regression models, predicting on held-out data, and comparing them with MAE, RMSE, and R². The stronger model is the one with lower prediction errors and a higher R², interpreted in light of the dataset and limitations.

## References
1. Notre Dame Global Adaptation Initiative (ND-GAIN), Country Index: https://gain.nd.edu/our-work/country-index/
2. Scikit-learn documentation, Linear Regression: https://scikit-learn.org/stable/modules/linear_model.html
3. Scikit-learn documentation, Random Forest Regression: https://scikit-learn.org/stable/modules/ensemble.html#random-forests
4. Scikit-learn documentation, Model evaluation: https://scikit-learn.org/stable/modules/model_evaluation.html


# Week 7: Exploring Pretrained Models

**Project:** Using Machine Learning to Support Climate Governance and Adaptation Decision-Making  
**Student:** Erick Munala Sifuna

## 11. Selected pretrained model: ClimateBERT

**Model name:** `climatebert/distilroberta-base-climate-f`  
**Hugging Face model page:** https://huggingface.co/climatebert/distilroberta-base-climate-f

ClimateBERT is a pretrained language model based on DistilRoBERTa. It was further pretrained on climate-related text, including research-paper abstracts, corporate reports, company reports, and general news. Its model card identifies it as a fill-mask model. It predicts a suitable word or token to replace a `[MASK]` token in a sentence.

### Input and output

- **Input:** English text containing a mask token, for example: “Climate change increases [MASK] risks.”
- **Output:** Ranked candidate words/tokens that could fill the masked position, together with scores.
- **Model task:** Masked-language modelling (fill-mask), not direct numerical climate-risk prediction.

## 12. Relevance to climate governance and adaptation research

ClimateBERT could support the analysis of climate-policy documents, adaptation plans, government reports, environmental assessments, and climate-related news. For example, it could help researchers explore the language used when discussing vulnerability, mitigation, adaptation, emissions, and climate finance. It may be useful as an initial text-analysis component when comparing how institutions describe climate risks and adaptation priorities.

In the existing ND-GAIN project, the supervised machine-learning models use numerical country indicators to predict a climate-vulnerability score. ClimateBERT works with text instead. It could therefore complement the numerical model if climate-policy documents or reports were added as a separate source of information. It does not directly replace the regression models or calculate the ND-GAIN vulnerability score.

## 13. Difference from the models developed in Week 5

The Week 5 project used Linear Regression and Random Forest Regression. These models were trained on the selected numerical features and the ND-GAIN target variable using a training/test split. Their purpose was to predict a continuous climate-vulnerability score and evaluate the predictions using MAE, RMSE, and R².

ClimateBERT is different because it is already pretrained on large collections of climate-related text. It processes language and performs a fill-mask task. It does not learn the ND-GAIN numerical relationship from the project dataset, and it does not automatically output a country's vulnerability score.

## 14. Could pretrained models, generative AI, or NLP complement the project?

Yes. NLP models such as ClimateBERT could complement the project by extracting or exploring information from climate-policy documents and reports. Generative AI could assist with summarising long policy documents or organising themes for further human review. The extracted text-based information could later be compared with numerical indicators, provided that the text variables are carefully defined and validated.

These tools should be used as supporting methods. The numerical regression analysis remains necessary for the original prediction task.

## 15. Limitations and precautions

1. **Task limitation:** The selected checkpoint is a fill-mask model, not a ready-made climate-vulnerability scoring model.
2. **Language limitation:** Its usefulness may be reduced for documents written in languages or local contexts not adequately represented in its training data.
3. **Context and bias:** Predictions reflect patterns in training text and may reproduce omissions or biases in those sources.
4. **Interpretation:** A plausible predicted word is not evidence that a climate claim is scientifically correct.
5. **Local relevance:** Kenyan and African policy documents should be checked to ensure that the model represents local terminology and priorities.
6. **Validation:** Any extracted information should be checked against original documents and expert knowledge.
7. **Privacy and governance:** Sensitive or unpublished policy documents should not be uploaded to external services without permission.

## 16. Optional demonstration: fill-mask prediction

The following demonstration uses a short illustrative sentence. It does not train or fine-tune ClimateBERT and does not make a scientific claim.



In [ ]:
# Install the required package if it is not already installed.
!pip -q install transformers torch

from transformers import pipeline

climate_fill_mask = pipeline(
    "fill-mask",
    model="climatebert/distilroberta-base-climate-f"
)

example_text = "Climate change increases [MASK] risks."
demo_results = climate_fill_mask(example_text, top_k=5)

for result in demo_results:
    print(result["token_str"].strip(), round(result["score"], 4))



### Demonstration interpretation

The model returns candidate words for the masked position and a score for each candidate. The scores indicate the model's relative preference among the candidate completions; they are not probabilities that a climate statement is true. The example is only a demonstration of the model's text-completion function.

## 17. Conclusion

ClimateBERT is relevant to climate governance because it has been pretrained on climate-related text and can support exploratory analysis of policy documents, reports, and climate communication. Its fill-mask task differs from the Week 5 regression models, which use numerical country indicators to predict a continuous vulnerability score. The two approaches could complement one another if text-based evidence is added carefully, but ClimateBERT should not be treated as a substitute for validated numerical modelling or expert climate assessment.

## References

1. ClimateBERT model card, Hugging Face: https://huggingface.co/climatebert/distilroberta-base-climate-f
2. Webersinke, N., Kraus, M., Bingler, J. & Leippold, M. (2022). *ClimateBERT: A Pretrained Language Model for Climate-Related Text*. https://doi.org/10.48550/arXiv.2212.13631
3. ND-GAIN Country Index: https://gain.nd.edu/our-work/country-index/
